# CS3807 – Deep Learning Laboratory
## Experiment 5 – Comprehensive CNN Study
**Shiv Nadar University Chennai | B.Tech AI & DS | Semester V | AY 2026-27**

---
### Topics Covered
1. Weight Initialisation
2. Regularisation & Overfitting
3. Batch Normalisation
4. Optimisation Algorithms
5. CNN Hyperparameter Tuning
6. Transfer Learning & Fine-Tuning (MobileNetV2)
7. 5-Fold Cross-Validation for Model Selection
8. Final Model Evaluation
9. Additional Exercise (2 new configurations)

---
> **Read before running:**  
> - Each section is self-contained but shares the dataset helpers defined in Section 1.  
> - Run cells top-to-bottom the first time.  
> - Results are printed/plotted inline; copy them into `results_template.txt` afterwards.  
> - Keep `SEED = 42` unchanged to reproduce results.

## Section 1 – Imports & Global Configuration

In [ ]:
# ── Standard library ────────────────────────────────────────────────────────
import os
import time
import warnings
warnings.filterwarnings('ignore')

# ── Numerical / plotting ─────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.model_selection import KFold

# ── TensorFlow / Keras ───────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import tensorflow_datasets as tfds

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# ── Global training defaults (change here, takes effect everywhere) ───────────
IMG_SIZE    = 224          # MobileNetV2 expects 224×224
BATCH_SIZE  = 32           # default batch size
EPOCHS_FAST = 10           # short runs for ablation studies
EPOCHS_FULL = 20           # longer run for fine-tuning / final model
NUM_CLASSES = 37           # Oxford-IIIT Pet has 37 breeds
AUTOTUNE    = tf.data.AUTOTUNE

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))
print(f'Working with {NUM_CLASSES} classes, images resized to {IMG_SIZE}×{IMG_SIZE}')

### 1.1 Dataset Loading & Preprocessing

We use **tensorflow_datasets** to download the Oxford-IIIT Pet dataset automatically.  
Images are resized to 224×224 and normalised using `preprocess_input` from MobileNetV2.

In [ ]:
# ── Load Oxford-IIIT Pet from tensorflow_datasets ───────────────────────────
# First run will download ~800 MB; subsequent runs use the local cache.
(ds_train_raw, ds_val_raw, ds_test_raw), ds_info = tfds.load(
    'oxford_iiit_pet',
    split=['train[:80%]', 'train[80%:]', 'test'],
    with_info=True,
    as_supervised=True,  # returns (image, label) tuples
)

print('Train samples :', len(ds_train_raw))
print('Val   samples :', len(ds_val_raw))
print('Test  samples :', len(ds_test_raw))

In [ ]:
# ── Preprocessing pipeline ──────────────────────────────────────────────────
def preprocess(image, label):
    """Resize → cast → MobileNetV2 normalise (scales to [-1, 1])."""
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)  # MobileNetV2-specific normalisation
    return image, label

def augment(image, label):
    """Light augmentation used only on the training split."""
    image, label = preprocess(image, label)
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.1)
    return image, label

def make_dataset(ds, augment_flag=False, shuffle=False):
    """Build a ready-to-train tf.data.Dataset."""
    if shuffle:
        ds = ds.shuffle(buffer_size=1000, seed=SEED)
    if augment_flag:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    else:
        ds = ds.map(preprocess, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

# Build default datasets (reused across sections)
train_ds = make_dataset(ds_train_raw, augment_flag=True, shuffle=True)
val_ds   = make_dataset(ds_val_raw)
test_ds  = make_dataset(ds_test_raw)

print('Dataset pipelines ready.')

In [ ]:
# ── Visualise a few training samples to sanity-check ────────────────────────
class_names = ds_info.features['label'].names

sample_batch = next(iter(train_ds))
images, labels = sample_batch

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle('Sample images from Oxford-IIIT Pet (after preprocessing)', fontsize=13)
for i, ax in enumerate(axes.flat):
    # Reverse MobileNetV2 normalisation for display: map [-1,1] → [0,1]
    img_display = (images[i].numpy() + 1.0) / 2.0
    img_display = np.clip(img_display, 0, 1)
    ax.imshow(img_display)
    ax.set_title(class_names[labels[i].numpy()], fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

### 1.2 Shared Model Builder

A single helper function creates a MobileNetV2-based classifier.  
Parameters controlling initialisation, regularisation, dropout and freezing are exposed  
so every experiment can reuse this function without copy-pasting architecture code.

In [ ]:
def build_model(
    num_classes      = NUM_CLASSES,
    freeze_base      = True,      # True = feature extraction; False = fine-tune
    unfreeze_from    = None,      # layer name to start unfreezing (fine-tune)
    dropout_rate     = 0.3,
    l2_lambda        = 0.0,       # 0 = no L2 regularisation
    use_batchnorm    = True,
    kernel_init      = 'glorot_uniform',  # weight initialisation for dense head
):
    """
    Builds a MobileNetV2 classifier.

    Architecture:
        MobileNetV2 backbone (ImageNet weights)
        → GlobalAveragePooling2D
        → [optional BatchNorm]
        → [optional Dropout]
        → Dense(256, relu)  with L2 and chosen initialiser
        → Dense(num_classes, softmax)
    """
    # ── 1. Load MobileNetV2 backbone (no top classifier) ─────────────────────
    base = MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,          # drop ImageNet head
        weights='imagenet',         # pretrained feature extractor
    )

    # ── 2. Freeze / unfreeze strategy ────────────────────────────────────────
    base.trainable = not freeze_base  # freeze everything first
    if not freeze_base and unfreeze_from is not None:
        # Selective unfreezing: freeze layers before unfreeze_from
        set_trainable = False
        for layer in base.layers:
            if layer.name == unfreeze_from:
                set_trainable = True
            layer.trainable = set_trainable

    # ── 3. Build classification head ─────────────────────────────────────────
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False)   # training=False keeps BN frozen
    x = layers.GlobalAveragePooling2D()(x)

    if use_batchnorm:
        x = layers.BatchNormalization()(x)

    if dropout_rate > 0:
        x = layers.Dropout(dropout_rate, seed=SEED)(x)

    # Dense hidden layer
    reg = regularizers.l2(l2_lambda) if l2_lambda > 0 else None
    x = layers.Dense(
        256,
        activation='relu',
        kernel_regularizer=reg,
        kernel_initializer=kernel_init,
    )(x)

    # Output layer
    outputs = layers.Dense(
        num_classes,
        activation='softmax',
        kernel_initializer=kernel_init,
    )(x)

    model = keras.Model(inputs, outputs)
    return model

print('build_model() helper defined.')

In [ ]:
# ── Utility: plot training curves ────────────────────────────────────────────
def plot_curves(histories, labels, metric='accuracy', title='', figsize=(10, 4)):
    """
    Plot training and/or validation curves from a list of History objects.

    Args:
        histories : list of keras History objects (or dicts with .history)
        labels    : list of string labels for each curve
        metric    : 'accuracy' or 'loss'
        title     : plot title
    """
    val_metric = 'val_' + metric
    fig, axes = plt.subplots(1, 2, figsize=figsize)

    for h, label in zip(histories, labels):
        hist = h.history if hasattr(h, 'history') else h
        epochs = range(1, len(hist[metric]) + 1)
        axes[0].plot(epochs, hist[metric],    label=label)
        axes[1].plot(epochs, hist[val_metric], label=label)

    axes[0].set_title(f'Training {metric.capitalize()}')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel(metric.capitalize())
    axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

    axes[1].set_title(f'Validation {metric.capitalize()}')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel(metric.capitalize())
    axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

    fig.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

print('plot_curves() helper defined.')

---
## Section 2 – Weight Initialisation

**Theory:**  
Weight initialisation sets the starting point for gradient descent. Poor choices can cause:
- **Vanishing gradients** (weights too small → gradients shrink toward zero)
- **Exploding gradients** (weights too large → gradients grow uncontrollably)
- **Symmetry breaking failure** (all-zero init → all neurons learn identically)

| Strategy | Formula | Best for |
|---|---|---|
| Zero | w = 0 | — (almost always wrong) |
| Random Normal | w ~ N(0, 0.01) | quick sanity checks |
| Xavier/Glorot | w ~ U(-√(6/(fan_in+fan_out)), …) | sigmoid / tanh |
| He | w ~ N(0, √(2/fan_in)) | ReLU / variants |

We freeze the MobileNetV2 backbone and only vary the initialisation of the **dense head**,  
so differences arise purely from the head initialisation strategy.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
init_strategies = {
    'Zero'         : tf.initializers.Zeros(),
    'Random Normal': 'random_normal',      # small N(0,0.05)
    'Xavier/Glorot': 'glorot_uniform',     # default Keras init
    'He Normal'    : 'he_normal',
}

init_histories = {}   # store History objects for later plotting

for name, init in init_strategies.items():
    print(f'\n>>> Training with {name} initialisation ...')
    model = build_model(
        freeze_base   = True,
        dropout_rate  = 0.3,
        kernel_init   = init,
        use_batchnorm = True,
    )
    model.compile(
        optimizer = keras.optimizers.Adam(1e-3),
        loss      = 'sparse_categorical_crossentropy',
        metrics   = ['accuracy'],
    )
    history = model.fit(
        train_ds,
        epochs          = EPOCHS_FAST,
        validation_data = val_ds,
        verbose         = 1,
    )
    init_histories[name] = history
    # Print final validation accuracy for quick comparison
    best_val = max(history.history['val_accuracy'])
    print(f'  Best val accuracy ({name}): {best_val:.4f}')

In [ ]:
# ── Plot 1: Training Loss vs Epoch ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
for name, h in init_histories.items():
    epochs = range(1, len(h.history['loss']) + 1)
    ax.plot(epochs, h.history['loss'], label=name, marker='o', markersize=3)
ax.set_title('Plot 1 – Training Loss vs Epoch\n(Different Weight Initialisations)', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Training Loss')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plot1_init_training_loss.png', dpi=150)
plt.show()
print('Plot 1 saved.')

In [ ]:
# ── Plot 2: Validation Accuracy vs Epoch ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
for name, h in init_histories.items():
    epochs = range(1, len(h.history['val_accuracy']) + 1)
    ax.plot(epochs, h.history['val_accuracy'], label=name, marker='o', markersize=3)
ax.set_title('Plot 2 – Validation Accuracy vs Epoch\n(Different Weight Initialisations)', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Accuracy (%)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plot2_init_val_accuracy.png', dpi=150)
plt.show()
print('Plot 2 saved.')

### Inference – Plots 1 & 2 (Initialisation)

> **TODO:** Fill this in after running the cells above.

**Plot 1 – Training Loss:**  
*What the plot shows:* Training loss curves for four initialisation strategies over 10 epochs.  
*Trend:* [e.g., Zero init does not decrease; He and Glorot converge quickly; Random Normal is slower.]  
*Why:* [e.g., Zero init breaks symmetry – all neurons receive identical gradients. He init is suited to ReLU, so it converges faster in ReLU-headed networks.]

**Plot 2 – Validation Accuracy:**  
*What the plot shows:* Validation accuracy per epoch for each strategy.  
*Trend:* [e.g., Glorot and He reach highest accuracy; Zero stays near chance.]  
*Why:* [e.g., Appropriate variance in initial weights lets the network explore the loss surface effectively from the first epoch.]

---
## Section 3 – Regularisation & Overfitting

**Theory:**  
Overfitting = the model memorises training data and fails to generalise.  
Symptoms: training accuracy >> validation accuracy; validation loss increases while training loss falls.

| Technique | Mechanism |
|---|---|
| No regularisation | Baseline – model free to overfit |
| L2 (weight decay) | Penalises large weights; adds λ‖w‖² to loss |
| Dropout | Randomly zeroes activations at rate p; reduces co-adaptation |
| Batch Normalisation | Normalises activations; acts as implicit regulariser |

In [ ]:
# ── Regularisation configurations to compare ─────────────────────────────────
reg_configs = {
    'No Regularisation': dict(dropout_rate=0.0, l2_lambda=0.0, use_batchnorm=False),
    'L2 (λ=1e-4)'      : dict(dropout_rate=0.0, l2_lambda=1e-4, use_batchnorm=False),
    'Dropout (p=0.5)'  : dict(dropout_rate=0.5, l2_lambda=0.0,  use_batchnorm=False),
    'Batch Norm'       : dict(dropout_rate=0.0, l2_lambda=0.0,  use_batchnorm=True),
}

reg_histories = {}

for name, cfg in reg_configs.items():
    print(f'\n>>> Training with {name} ...')
    model = build_model(freeze_base=True, kernel_init='he_normal', **cfg)
    model.compile(
        optimizer = keras.optimizers.Adam(1e-3),
        loss      = 'sparse_categorical_crossentropy',
        metrics   = ['accuracy'],
    )
    history = model.fit(
        train_ds,
        epochs          = EPOCHS_FAST,
        validation_data = val_ds,
        verbose         = 1,
    )
    reg_histories[name] = history
    best_val = max(history.history['val_accuracy'])
    print(f'  Best val accuracy ({name}): {best_val:.4f}')

In [ ]:
# ── Plot 3: Training vs Validation Accuracy (gap = generalisation gap) ────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flat
for ax, (name, h) in zip(axes, reg_histories.items()):
    epochs = range(1, len(h.history['accuracy']) + 1)
    ax.plot(epochs, h.history['accuracy'],     label='Train', linewidth=2)
    ax.plot(epochs, h.history['val_accuracy'], label='Val',   linewidth=2, linestyle='--')
    ax.fill_between(
        epochs,
        h.history['accuracy'],
        h.history['val_accuracy'],
        alpha=0.15, label='Generalisation gap'
    )
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
fig.suptitle('Plot 3 – Training vs Validation Accuracy per Regularisation Strategy', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot3_reg_accuracy.png', dpi=150)
plt.show()
print('Plot 3 saved.')

In [ ]:
# ── Plot 4: Training vs Validation Loss ──────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flat
for ax, (name, h) in zip(axes, reg_histories.items()):
    epochs = range(1, len(h.history['loss']) + 1)
    ax.plot(epochs, h.history['loss'],     label='Train Loss', linewidth=2)
    ax.plot(epochs, h.history['val_loss'], label='Val Loss',   linewidth=2, linestyle='--')
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
fig.suptitle('Plot 4 – Training vs Validation Loss per Regularisation Strategy', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot4_reg_loss.png', dpi=150)
plt.show()
print('Plot 4 saved.')

### Inference – Plots 3 & 4 (Regularisation)

**Plot 3 – Accuracy:**  
*What:* Training and validation accuracy per epoch with shaded generalisation gap.  
*Trend:* [e.g., No-reg model shows widening gap → overfitting. Dropout and BN reduce the gap.]  
*Why:* [Dropout prevents co-adaptation; BN reduces internal covariate shift, acting as implicit regulariser.]

**Plot 4 – Loss:**  
*What:* Whether validation loss rises while training loss falls (classic overfitting signature).  
*Trend:* [e.g., Without regularisation, val-loss starts increasing around epoch 6.]  
*Why:* [The model memorises training examples rather than learning generalisable features.]

---
## Section 4 – Batch Normalisation (Numerical Example + Ablation)

**Formula recap:**
$$\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}, \quad y_i = \gamma \hat{x}_i + \beta$$

where $\mu_B$ and $\sigma_B^2$ are computed per mini-batch, and $\gamma$, $\beta$ are learnable.

In [ ]:
# ── Numerical BN example from the assignment sheet ───────────────────────────
import math

x = np.array([2.0, 4.0, 6.0, 8.0])
eps = 1e-7          # tiny epsilon for numerical stability
gamma, beta = 1.0, 0.0   # default learnable params

# Step 1: batch mean
mu_B = np.mean(x)
print(f'Step 1 – Batch mean (μB)    = {mu_B}')

# Step 2: batch variance
var_B = np.var(x)
print(f'Step 2 – Batch variance (σ²B) = {var_B}')

# Step 3: normalise
x_hat = (x - mu_B) / math.sqrt(var_B + eps)
print(f'Step 3 – Normalised  (x̂)   = {np.round(x_hat, 3)}')

# Step 4: scale & shift
y = gamma * x_hat + beta
print(f'Step 4 – Output (γ=1,β=0)   = {np.round(y, 3)}')
print()
print('Inference: BN produces ≈zero-mean, unit-variance activations.')
print('Learnable γ and β let the network undo normalisation if needed.')

In [ ]:
# ── Plot 5: With BN vs Without BN ────────────────────────────────────────────
# Already computed in Section 3; extract from reg_histories
h_no_bn  = reg_histories['No Regularisation']
h_with_bn = reg_histories['Batch Norm']

fig, ax = plt.subplots(figsize=(8, 5))
epochs = range(1, len(h_no_bn.history['val_accuracy']) + 1)
ax.plot(epochs, h_no_bn.history['val_accuracy'],  label='Without BN', linewidth=2, color='tomato')
ax.plot(epochs, h_with_bn.history['val_accuracy'], label='With BN',    linewidth=2, color='steelblue')
ax.set_title('Plot 5 – Validation Accuracy: With vs Without Batch Normalisation', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plot5_batchnorm.png', dpi=150)
plt.show()
print('Plot 5 saved.')

### Inference – Plot 5 (Batch Normalisation)

*What:* Comparison of validation accuracy with and without BN over 10 epochs.  
*Trend:* [e.g., With BN, accuracy is higher and less noisy from the start.]  
*Why:* [BN stabilises activations at each layer, reducing internal covariate shift, which allows a higher learning rate and faster convergence.]

---
## Section 5 – Optimisation Algorithms

| Optimiser | Key idea |
|---|---|
| SGD | Plain gradient descent with fixed LR |
| Momentum | Accumulates a velocity vector to smooth updates |
| RMSProp | Divides LR by a running mean of squared gradients |
| Adam | Combines Momentum + RMSProp; maintains first & second moment estimates |

In [ ]:
# ── Define optimisers ─────────────────────────────────────────────────────────
optimisers = {
    'SGD'      : keras.optimizers.SGD(learning_rate=1e-3),
    'Momentum' : keras.optimizers.SGD(learning_rate=1e-3, momentum=0.9),
    'RMSProp'  : keras.optimizers.RMSprop(learning_rate=1e-3),
    'Adam'     : keras.optimizers.Adam(learning_rate=1e-3),
}

opt_histories = {}
opt_results   = {}   # store final metrics for the comparison table

for name, opt in optimisers.items():
    print(f'\n>>> Training with {name} ...')
    start = time.time()
    model = build_model(freeze_base=True, kernel_init='he_normal',
                        dropout_rate=0.3, use_batchnorm=True)
    model.compile(
        optimizer = opt,
        loss      = 'sparse_categorical_crossentropy',
        metrics   = ['accuracy'],
    )
    history = model.fit(
        train_ds,
        epochs          = EPOCHS_FAST,
        validation_data = val_ds,
        verbose         = 1,
    )
    elapsed = time.time() - start
    best_val = max(history.history['val_accuracy'])
    best_epoch = history.history['val_accuracy'].index(best_val) + 1
    final_loss = history.history['loss'][-1]
    opt_histories[name] = history
    opt_results[name] = {
        'Final Loss'   : round(final_loss, 4),
        'Best Val Acc' : round(best_val, 4),
        'Epoch'        : best_epoch,
        'Time (s)'     : round(elapsed, 1),
    }
    print(f'  → Loss={final_loss:.4f}  ValAcc={best_val:.4f}  Epoch={best_epoch}  Time={elapsed:.0f}s')

In [ ]:
# ── Print summary table ───────────────────────────────────────────────────────
print('\n' + '='*70)
print(f'{"Optimiser":<12} {"Final Loss":<14} {"Best Val Acc":<16} {"Epoch":<10} {"Time (s)"}')
print('='*70)
for name, r in opt_results.items():
    print(f'{name:<12} {r["Final Loss"]:<14} {r["Best Val Acc"]:<16} {r["Epoch"]:<10} {r["Time (s)"]}')
print('='*70)

In [ ]:
# ── Plot 6: Training Loss vs Epoch (all optimisers) ──────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
for name, h in opt_histories.items():
    epochs = range(1, len(h.history['loss']) + 1)
    ax.plot(epochs, h.history['loss'], label=name, marker='o', markersize=3)
ax.set_title('Plot 6 – Training Loss vs Epoch (Different Optimisers)', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Training Loss')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plot6_opt_train_loss.png', dpi=150)
plt.show()
print('Plot 6 saved.')

In [ ]:
# ── Plot 7: Validation Accuracy vs Epoch (all optimisers) ────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
for name, h in opt_histories.items():
    epochs = range(1, len(h.history['val_accuracy']) + 1)
    ax.plot(epochs, h.history['val_accuracy'], label=name, marker='o', markersize=3)
ax.set_title('Plot 7 – Validation Accuracy vs Epoch (Different Optimisers)', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plot7_opt_val_accuracy.png', dpi=150)
plt.show()
print('Plot 7 saved.')

### Inference – Plots 6 & 7 (Optimisers)

**Plot 6 – Training Loss:**  
*What:* Convergence speed of each optimiser measured by training loss.  
*Trend:* [e.g., Adam and RMSProp converge faster; plain SGD is slowest.]  
*Why:* [Adam adapts learning rate per parameter using moment estimates, reducing oscillation.]

**Plot 7 – Validation Accuracy:**  
*What:* Generalisation quality per optimiser.  
*Trend:* [e.g., Adam achieves highest val accuracy; SGD may catch up given more epochs.]  
*Why:* [Adaptive methods escape flat regions and saddle points more effectively in deep networks.]

---
## Section 6 – CNN Hyperparameter Tuning

**Rule:** Change **one hyperparameter at a time** while keeping all others fixed.

**Output dimension formula:**
$$O = \left\lfloor \frac{N + 2P - K}{S} \right\rfloor + 1$$

| Symbol | Meaning |
|---|---|
| N | Input size |
| K | Kernel size |
| P | Padding |
| S | Stride |

In [ ]:
# ── 6.1 Learning Rate sweep ──────────────────────────────────────────────────
lr_values   = [0.01, 0.001, 0.0001, 0.00001]
lr_results  = {}   # name → best val accuracy

print('=== Learning Rate Sweep ===')
for lr in lr_values:
    name = f'LR={lr}'
    model = build_model(freeze_base=True, kernel_init='he_normal',
                        dropout_rate=0.3, use_batchnorm=True)
    model.compile(
        optimizer = keras.optimizers.Adam(lr),
        loss      = 'sparse_categorical_crossentropy',
        metrics   = ['accuracy'],
    )
    h = model.fit(train_ds, epochs=EPOCHS_FAST,
                  validation_data=val_ds, verbose=0)
    best_val = max(h.history['val_accuracy'])
    lr_results[lr] = best_val
    print(f'  LR={lr:<8} → Best Val Acc = {best_val:.4f}')

In [ ]:
# ── Plot 8: Learning Rate vs Validation Accuracy ─────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
lrs  = list(lr_results.keys())
accs = list(lr_results.values())
ax.plot(lrs, accs, marker='o', linewidth=2, color='darkorange')
ax.set_xscale('log')
ax.set_title('Plot 8 – Learning Rate vs Validation Accuracy', fontweight='bold')
ax.set_xlabel('Learning Rate (log scale)')
ax.set_ylabel('Best Validation Accuracy')
ax.grid(True, alpha=0.3)
for lr, acc in zip(lrs, accs):
    ax.annotate(f'{acc:.3f}', (lr, acc), textcoords='offset points', xytext=(0, 8), fontsize=9)
plt.tight_layout()
plt.savefig('plot8_lr_sweep.png', dpi=150)
plt.show()
print('Plot 8 saved.')

In [ ]:
# ── 6.2 Batch Size sweep ─────────────────────────────────────────────────────
batch_sizes  = [16, 32, 64]
bs_results   = {}

print('=== Batch Size Sweep ===')
for bs in batch_sizes:
    # Rebuild dataset with different batch size
    tr_ds_bs = make_dataset(ds_train_raw, augment_flag=True, shuffle=True)
    tr_ds_bs = ds_train_raw.map(augment, num_parallel_calls=AUTOTUNE).batch(bs).prefetch(AUTOTUNE)
    va_ds_bs = ds_val_raw.map(preprocess, num_parallel_calls=AUTOTUNE).batch(bs).prefetch(AUTOTUNE)

    model = build_model(freeze_base=True, kernel_init='he_normal',
                        dropout_rate=0.3, use_batchnorm=True)
    model.compile(
        optimizer = keras.optimizers.Adam(1e-3),
        loss      = 'sparse_categorical_crossentropy',
        metrics   = ['accuracy'],
    )
    h = model.fit(tr_ds_bs, epochs=EPOCHS_FAST,
                  validation_data=va_ds_bs, verbose=0)
    best_val = max(h.history['val_accuracy'])
    bs_results[bs] = best_val
    print(f'  BatchSize={bs:<4} → Best Val Acc = {best_val:.4f}')

In [ ]:
# ── Plot 9: Batch Size vs Validation Accuracy ─────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
bss  = list(bs_results.keys())
accs = list(bs_results.values())
ax.bar([str(b) for b in bss], accs, color=['steelblue', 'seagreen', 'tomato'], width=0.4)
ax.set_title('Plot 9 – Batch Size vs Validation Accuracy', fontweight='bold')
ax.set_xlabel('Batch Size')
ax.set_ylabel('Best Validation Accuracy')
ax.set_ylim(0, 1)
for i, (b, a) in enumerate(zip(bss, accs)):
    ax.text(i, a + 0.01, f'{a:.3f}', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('plot9_bs_sweep.png', dpi=150)
plt.show()
print('Plot 9 saved.')

In [ ]:
# ── 6.3 Dropout Rate sweep ───────────────────────────────────────────────────
dropout_rates = [0.0, 0.25, 0.5]
dr_results    = {}

print('=== Dropout Rate Sweep ===')
for dr in dropout_rates:
    model = build_model(freeze_base=True, kernel_init='he_normal',
                        dropout_rate=dr, use_batchnorm=True)
    model.compile(
        optimizer = keras.optimizers.Adam(1e-3),
        loss      = 'sparse_categorical_crossentropy',
        metrics   = ['accuracy'],
    )
    h = model.fit(train_ds, epochs=EPOCHS_FAST,
                  validation_data=val_ds, verbose=0)
    best_val = max(h.history['val_accuracy'])
    dr_results[dr] = best_val
    print(f'  Dropout={dr:<5} → Best Val Acc = {best_val:.4f}')

In [ ]:
# ── Plot 10: Dropout Rate vs Validation Accuracy ─────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
drs  = list(dr_results.keys())
accs = list(dr_results.values())
ax.plot(drs, accs, marker='D', linewidth=2, color='mediumpurple')
ax.set_title('Plot 10 – Dropout Rate vs Validation Accuracy', fontweight='bold')
ax.set_xlabel('Dropout Rate')
ax.set_ylabel('Best Validation Accuracy')
ax.grid(True, alpha=0.3)
for dr, acc in zip(drs, accs):
    ax.annotate(f'{acc:.3f}', (dr, acc), textcoords='offset points', xytext=(0, 8), fontsize=9)
plt.tight_layout()
plt.savefig('plot10_dropout_sweep.png', dpi=150)
plt.show()
print('Plot 10 saved.')

### Inference – Plots 8, 9, 10 (Hyperparameter Tuning)

**Plot 8 – Learning Rate:**  
*What:* Effect of LR magnitude on final validation accuracy.  
*Trend:* [e.g., LR=0.001 peaks; LR=0.01 overshoots; LR=1e-5 underfits.]  
*Why:* [Too large → gradients oscillate; too small → slow convergence / stuck in local minima.]

**Plot 9 – Batch Size:**  
*What:* Accuracy change as batch size grows.  
*Trend:* [e.g., Smaller batches generise better due to noisier gradient estimates.]  
*Why:* [Small batches introduce stochasticity acting as implicit regularisation (sharp minima avoided).]

**Plot 10 – Dropout Rate:**  
*What:* Accuracy vs dropout probability.  
*Trend:* [e.g., Moderate dropout (0.25–0.5) outperforms no dropout.]  
*Why:* [Dropout forces the network not to rely on specific neurons, improving robustness.]

---
## Section 7 – Transfer Learning & Fine-Tuning

| Case | Strategy |
|---|---|
| **A – Feature Extraction** | Freeze entire MobileNetV2 backbone; train only the new head |
| **B – Fine-Tuning**       | Unfreeze upper backbone layers; train with a very small LR |

**Why small LR for fine-tuning?**  
The pretrained weights already encode ImageNet features. A large LR would destroy them.  
A small LR (1e-5) nudges the weights gently toward the pet domain without catastrophic forgetting.

In [ ]:
# ── Case A: Feature Extraction ───────────────────────────────────────────────
print('>>> Case A: Feature Extraction (frozen base) ...')
model_fe = build_model(
    freeze_base   = True,
    dropout_rate  = 0.3,
    use_batchnorm = True,
    kernel_init   = 'he_normal',
)
model_fe.compile(
    optimizer = keras.optimizers.Adam(1e-3),
    loss      = 'sparse_categorical_crossentropy',
    metrics   = ['accuracy'],
)
history_fe = model_fe.fit(
    train_ds,
    epochs          = EPOCHS_FAST,
    validation_data = val_ds,
    verbose         = 1,
)
print(f'Feature Extraction – Best Val Acc: {max(history_fe.history["val_accuracy"]):.4f}')

In [ ]:
# ── Case B: Fine-Tuning (unfreeze last ~30 layers of backbone) ───────────────
print('\n>>> Case B: Fine-Tuning (unfreeze top layers, LR=1e-5) ...')

# Start from the feature-extraction model (already warm-started)
# Unfreeze from the 'block_16_expand' layer onward
base_model = model_fe.layers[1]   # MobileNetV2 is the second layer
base_model.trainable = True

# Freeze everything except the last 30 layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Recompile with a much smaller learning rate
model_fe.compile(
    optimizer = keras.optimizers.Adam(1e-5),   # SMALL LR – critical for fine-tuning
    loss      = 'sparse_categorical_crossentropy',
    metrics   = ['accuracy'],
)

trainable_count = sum(tf.size(v).numpy() for v in model_fe.trainable_variables)
print(f'Trainable parameters after unfreezing: {trainable_count:,}')

history_ft = model_fe.fit(
    train_ds,
    epochs          = EPOCHS_FAST,
    validation_data = val_ds,
    verbose         = 1,
)
print(f'Fine-Tuning – Best Val Acc: {max(history_ft.history["val_accuracy"]):.4f}')

In [ ]:
# ── Plot 11: Feature Extraction vs Fine-Tuning (val accuracy) ────────────────
fig, ax = plt.subplots(figsize=(8, 5))
e_fe = range(1, len(history_fe.history['val_accuracy']) + 1)
e_ft = range(1, len(history_ft.history['val_accuracy']) + 1)
ax.plot(e_fe, history_fe.history['val_accuracy'], label='Feature Extraction', linewidth=2)
ax.plot(e_ft, history_ft.history['val_accuracy'], label='Fine-Tuning',        linewidth=2, linestyle='--')
ax.set_title('Plot 11 – Feature Extraction vs Fine-Tuning\n(Validation Accuracy)', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plot11_fe_vs_ft.png', dpi=150)
plt.show()
print('Plot 11 saved.')

In [ ]:
# ── Plot 12: Training and Validation Loss (before vs after fine-tuning) ───────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, h, title in [
    (axes[0], history_fe, 'Before Fine-Tuning (Feature Extraction)'),
    (axes[1], history_ft, 'After Fine-Tuning'),
]:
    epochs = range(1, len(h.history['loss']) + 1)
    ax.plot(epochs, h.history['loss'],     label='Train Loss', linewidth=2)
    ax.plot(epochs, h.history['val_loss'], label='Val Loss',   linewidth=2, linestyle='--')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
fig.suptitle('Plot 12 – Loss Curves Before and After Fine-Tuning', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot12_loss_ft.png', dpi=150)
plt.show()
print('Plot 12 saved.')

### Inference – Plots 11 & 12 (Transfer Learning)

**Plot 11:**  
*What:* Comparison of validation accuracy between frozen-base and fine-tuned models.  
*Trend:* [e.g., Fine-tuning yields higher and more stable accuracy after warm-up.]  
*Why:* [Unfreezing domain-relevant upper layers allows the network to adapt ImageNet features to pet textures.]

**Plot 12:**  
*What:* Loss dynamics before and after fine-tuning.  
*Trend:* [e.g., After fine-tuning, both training and val loss decrease together – less gap.]  
*Why:* [Small LR during fine-tuning prevents sudden jumps that would destroy pretrained feature representations.]

---
## Section 8 – 5-Fold Cross-Validation for Model Selection

**Why K-Fold?**  
A single train-val split can give an overly optimistic or pessimistic estimate depending on which examples happen to fall in each split.  
K-Fold gives K estimates and reports **mean ± SD** – a more reliable and honest performance figure.

**Configurations evaluated (pick the most promising from Sections 2–7):**

| Config | LR | Dropout | Batch | BN | Fine-tune |
|---|---|---|---|---|---|
| C1 – Baseline | 1e-3 | 0.3 | 32 | Yes | No |
| C2 – Low LR   | 1e-4 | 0.3 | 32 | Yes | No |
| C3 – High Dropout | 1e-3 | 0.5 | 32 | Yes | No |
| C4 – Fine-Tune | 1e-5 | 0.3 | 32 | Yes | Yes |

In [ ]:
# ── Build the full training numpy arrays for KFold splitting ─────────────────
# We convert the training tf.data.Dataset into numpy arrays once.
# This is memory-intensive but necessary for sklearn KFold.
print('Converting training data to numpy arrays for KFold ...')
X_train_np = []
y_train_np = []
# Use the raw (non-batched) dataset for this
for img, lbl in ds_train_raw.map(preprocess):
    X_train_np.append(img.numpy())
    y_train_np.append(lbl.numpy())
X_train_np = np.array(X_train_np)
y_train_np = np.array(y_train_np)
print(f'X shape: {X_train_np.shape}, y shape: {y_train_np.shape}')

In [ ]:
# ── Cross-validation runner ───────────────────────────────────────────────────
def run_kfold(
    X, y, config_fn, n_splits=5,
    epochs=EPOCHS_FAST, batch_size=BATCH_SIZE
):
    """
    Runs K-fold cross-validation.

    Args:
        X         : numpy array of images
        y         : numpy array of labels
        config_fn : callable() → (compiled keras model)
        n_splits  : number of folds (5)
    Returns:
        fold_accs : list of validation accuracies per fold
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    fold_accs = []

    for fold_idx, (tr_idx, va_idx) in enumerate(kf.split(X), 1):
        print(f'  Fold {fold_idx}/{n_splits} ...', end=' ')
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        # Build a fresh model for each fold
        model = config_fn()

        # Create tf.data pipelines from numpy
        tr_ds_fold = (
            tf.data.Dataset.from_tensor_slices((X_tr, y_tr))
            .shuffle(buffer_size=len(X_tr), seed=SEED)
            .batch(batch_size)
            .prefetch(AUTOTUNE)
        )
        va_ds_fold = (
            tf.data.Dataset.from_tensor_slices((X_va, y_va))
            .batch(batch_size)
            .prefetch(AUTOTUNE)
        )

        h = model.fit(tr_ds_fold, epochs=epochs,
                      validation_data=va_ds_fold, verbose=0)
        best_va = max(h.history['val_accuracy'])
        fold_accs.append(best_va)
        print(f'val_acc = {best_va:.4f}')

    return fold_accs

print('run_kfold() helper defined.')

In [ ]:
# ── Define four configurations as factory functions ───────────────────────────
def make_c1():  # Baseline
    m = build_model(freeze_base=True, dropout_rate=0.3,
                    use_batchnorm=True, kernel_init='he_normal')
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

def make_c2():  # Low LR
    m = build_model(freeze_base=True, dropout_rate=0.3,
                    use_batchnorm=True, kernel_init='he_normal')
    m.compile(optimizer=keras.optimizers.Adam(1e-4),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

def make_c3():  # High Dropout
    m = build_model(freeze_base=True, dropout_rate=0.5,
                    use_batchnorm=True, kernel_init='he_normal')
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

def make_c4():  # Fine-tune (unfreeze last 30 layers)
    m = build_model(freeze_base=False, dropout_rate=0.3,
                    use_batchnorm=True, kernel_init='he_normal')
    m.compile(optimizer=keras.optimizers.Adam(1e-5),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

configs = {
    'C1 – Baseline'    : make_c1,
    'C2 – Low LR'      : make_c2,
    'C3 – High Dropout': make_c3,
    'C4 – Fine-Tune'   : make_c4,
}

cv_results = {}   # config_name → list of 5 fold accuracies

for cfg_name, cfg_fn in configs.items():
    print(f'\n=== {cfg_name} ===')
    fold_accs = run_kfold(X_train_np, y_train_np,
                          config_fn=cfg_fn, n_splits=5, epochs=EPOCHS_FAST)
    cv_results[cfg_name] = fold_accs
    mean_acc = np.mean(fold_accs)
    std_acc  = np.std(fold_accs)
    print(f'  Result: {mean_acc:.4f} ± {std_acc:.4f}')

In [ ]:
# ── Print cross-validation summary table ─────────────────────────────────────
print('\n' + '='*75)
print(f'{"Config":<22} {"F1":<8} {"F2":<8} {"F3":<8} {"F4":<8} {"F5":<8} {"Mean":>7} {"±SD":>7}')
print('='*75)
for cfg_name, folds in cv_results.items():
    mean_a = np.mean(folds)
    std_a  = np.std(folds)
    fold_str = '  '.join([f'{f:.4f}' for f in folds])
    print(f'{cfg_name:<22} {fold_str}  {mean_a:.4f}  ±{std_a:.4f}')
print('='*75)

In [ ]:
# ── Plot 13: 5-Fold Cross-Validation Accuracy with error bars ─────────────────
cfg_names  = list(cv_results.keys())
means      = [np.mean(v) for v in cv_results.values()]
stds       = [np.std(v)  for v in cv_results.values()]

fig, ax = plt.subplots(figsize=(9, 6))
x_pos = range(len(cfg_names))
bars = ax.bar(x_pos, means, yerr=stds,
              capsize=8, color=['steelblue','seagreen','tomato','goldenrod'],
              width=0.5, alpha=0.85, error_kw=dict(elinewidth=2))
ax.set_xticks(x_pos)
ax.set_xticklabels(cfg_names, fontsize=9)
ax.set_title('Plot 13 – 5-Fold Cross-Validation Accuracy\n(Mean ± Standard Deviation)',
             fontweight='bold')
ax.set_xlabel('Hyperparameter Configuration')
ax.set_ylabel('Mean Validation Accuracy')
ax.set_ylim(0, 1)
# Annotate bars
for bar, m, s in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2, m + s + 0.01,
            f'{m:.3f}\n±{s:.3f}', ha='center', fontsize=8)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('plot13_kfold.png', dpi=150)
plt.show()
print('Plot 13 saved.')

In [ ]:
# ── Identify best configuration for final model ───────────────────────────────
# Select: highest mean accuracy AND lowest standard deviation (trade-off)
best_cfg_name = max(cv_results, key=lambda k: np.mean(cv_results[k]))
print(f'Best configuration (by mean CV accuracy): {best_cfg_name}')
print(f'  Mean = {np.mean(cv_results[best_cfg_name]):.4f}')
print(f'  SD   = {np.std(cv_results[best_cfg_name]):.4f}')

### Inference – Plot 13 (Cross-Validation)

*What:* Mean validation accuracy across 5 folds for each configuration, with ±SD error bars.  
*Trend:* [e.g., C4 (fine-tuning) has the highest mean but possibly larger SD.]  
*Why:* [Fine-tuning is more sensitive to the fold composition; a low-SD config may be preferable in practice.]  

**Important:** Both **mean** and **SD** matter.  
A configuration with slightly lower mean but much lower SD is often preferable in production.

---
## Section 9 – Final Model Evaluation

1. Retrain the best configuration on the **entire training set** (train + val combined).  
2. Evaluate once on the **held-out test set** (first and only time it is touched).  
3. Report all required metrics.

In [ ]:
# ── Combine train + val for final training ────────────────────────────────────
print('Building combined train+val dataset for final model ...')
full_train_ds = (
    ds_train_raw.concatenate(ds_val_raw)
    .map(augment, num_parallel_calls=AUTOTUNE)
    .shuffle(2000, seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# ── Build and train the final model ──────────────────────────────────────────
print('Training final model ...')
start = time.time()

# Use the best configuration factory; edit make_c4 / make_c1 etc as appropriate
best_config_fn = configs[best_cfg_name]
final_model = best_config_fn()

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
]

final_history = final_model.fit(
    full_train_ds,
    epochs            = EPOCHS_FULL,
    validation_data   = val_ds,   # used only for early stopping
    callbacks         = callbacks,
    verbose           = 1,
)
training_time = time.time() - start
print(f'\nFinal model training time: {training_time:.0f}s')

In [ ]:
# ── Evaluate on the test set ──────────────────────────────────────────────────
print('\n=== TEST SET EVALUATION ===')
test_loss, test_acc = final_model.evaluate(test_ds, verbose=1)
print(f'Test Loss     : {test_loss:.4f}')
print(f'Test Accuracy : {test_acc:.4f}')

In [ ]:
# ── Classification Report (Precision, Recall, F1) ────────────────────────────
print('\nGenerating predictions on test set ...')
y_true, y_pred = [], []
for imgs, lbls in test_ds:
    preds = final_model.predict(imgs, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(lbls.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

report = classification_report(y_true, y_pred,
                                target_names=class_names,
                                digits=4)
print(report)

# Overall macro averages
from sklearn.metrics import precision_score, recall_score, f1_score
precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
recall    = recall_score(y_true, y_pred,    average='macro', zero_division=0)
f1        = f1_score(y_true, y_pred,        average='macro', zero_division=0)
print(f'Macro Precision : {precision:.4f}')
print(f'Macro Recall    : {recall:.4f}')
print(f'Macro F1-Score  : {f1:.4f}')

In [ ]:
# ── Model parameter count ─────────────────────────────────────────────────────
total_params     = final_model.count_params()
trainable_params = sum(tf.size(v).numpy() for v in final_model.trainable_variables)
print(f'Total parameters    : {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

In [ ]:
# ── Plot 14: Confusion Matrix ─────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(20, 18))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, colorbar=True, xticks_rotation='vertical',
          cmap='Blues', values_format='d')
ax.set_title('Plot 14 – Confusion Matrix (Test Set)', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('plot14_confusion_matrix.png', dpi=150)
plt.show()
print('Plot 14 saved.')

# Best and worst classes
per_class_acc = cm.diagonal() / cm.sum(axis=1)
best_cls  = class_names[np.argmax(per_class_acc)]
worst_cls = class_names[np.argmin(per_class_acc)]
print(f'\nBest classified  : {best_cls}  ({per_class_acc.max():.3f})')
print(f'Worst classified : {worst_cls} ({per_class_acc.min():.3f})')

In [ ]:
# ── Plot 15 (Optional): Misclassified Images ─────────────────────────────────
# Find indices where prediction ≠ true label
misclassified = np.where(y_pred != y_true)[0]
print(f'Total misclassified: {len(misclassified)} / {len(y_true)}')

# Collect raw test images for display
test_images_all = []
for imgs, _ in test_ds:
    test_images_all.append(imgs.numpy())
test_images_all = np.concatenate(test_images_all, axis=0)

# Show up to 10 misclassified examples
n_show = min(10, len(misclassified))
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle('Plot 15 – Sample Misclassified Images', fontsize=13, fontweight='bold')
for ax, idx in zip(axes.flat, misclassified[:n_show]):
    img = (test_images_all[idx] + 1.0) / 2.0  # reverse MobileNetV2 normalisation
    img = np.clip(img, 0, 1)
    ax.imshow(img)
    ax.set_title(
        f'True: {class_names[y_true[idx]]}\nPred: {class_names[y_pred[idx]]}',
        fontsize=7, color='red'
    )
    ax.axis('off')
plt.tight_layout()
plt.savefig('plot15_misclassified.png', dpi=150)
plt.show()
print('Plot 15 saved.')

### Inference – Plots 14 & 15

**Plot 14 – Confusion Matrix:**  
*What:* Per-class prediction distribution on the test set.  
*Best class:* [e.g., 'Siamese' – high intra-class visual consistency.]  
*Most confused pair:* [e.g., 'Abyssinian' ↔ 'Egyptian Mau' – similar coat texture and colour.]

**Plot 15 – Misclassified Images:**  
*Observation:* [e.g., Many misclassifications involve unusual lighting, occlusion, or uncommon poses.]  
*Possible fix:* [More aggressive data augmentation; higher-resolution crops.]

---
## Section 10 – Additional Exercise (Two New Configurations)

**Requirement (Section 16 of assignment):**  
Select 2 new combinations of LR, dropout, batch size and fine-tuning strategy.  
Evaluate with 5-fold CV and compare to the selected best configuration.

| Config | LR | Dropout | Batch | Fine-tune Notes |
|---|---|---|---|---|
| C5 – Aggressive LR | 5e-4 | 0.4 | 16 | Frozen base |
| C6 – Full Fine-Tune | 1e-5 | 0.2 | 32 | Unfreeze all layers |

In [ ]:
# ── Configuration C5: Smaller batch + moderate dropout + mid LR ───────────────
def make_c5():
    """Aggressive LR with smaller batch (16) and moderate dropout."""
    m = build_model(freeze_base=True, dropout_rate=0.4,
                    use_batchnorm=True, kernel_init='he_normal')
    m.compile(
        optimizer = keras.optimizers.Adam(5e-4),
        loss      = 'sparse_categorical_crossentropy',
        metrics   = ['accuracy'],
    )
    return m

# ── Configuration C6: Full fine-tune (unfreeze all), very small LR ────────────
def make_c6():
    """Full backbone fine-tuning with very small LR and low dropout."""
    base = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3),
                       include_top=False, weights='imagenet')
    base.trainable = True   # unfreeze entire backbone
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2, seed=SEED)(x)
    x = layers.Dense(256, activation='relu', kernel_initializer='he_normal')(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    m = keras.Model(inputs, outputs)
    m.compile(
        optimizer = keras.optimizers.Adam(1e-5),
        loss      = 'sparse_categorical_crossentropy',
        metrics   = ['accuracy'],
    )
    return m

extra_configs = {
    'C5 – Agressive LR (bs=16)': make_c5,
    'C6 – Full Fine-Tune'       : make_c6,
}

extra_cv_results = {}

for cfg_name, cfg_fn in extra_configs.items():
    print(f'\n=== {cfg_name} ===')
    batch = 16 if 'bs=16' in cfg_name else BATCH_SIZE
    fold_accs = run_kfold(X_train_np, y_train_np,
                          config_fn=cfg_fn, n_splits=5,
                          epochs=EPOCHS_FAST, batch_size=batch)
    extra_cv_results[cfg_name] = fold_accs
    print(f'  {np.mean(fold_accs):.4f} ± {np.std(fold_accs):.4f}')

In [ ]:
# ── Compare all 6 configurations ─────────────────────────────────────────────
all_cv = {**cv_results, **extra_cv_results}

print('\n' + '='*80)
print(f'{"Config":<30} {"F1":<8} {"F2":<8} {"F3":<8} {"F4":<8} {"F5":<8} {"Mean":>7} {"±SD":>7}')
print('='*80)
for cfg_name, folds in all_cv.items():
    fold_str = '  '.join([f'{f:.4f}' for f in folds])
    print(f'{cfg_name:<30} {fold_str}  {np.mean(folds):.4f}  ±{np.std(folds):.4f}')
print('='*80)

# Final verdict
best_all = max(all_cv, key=lambda k: np.mean(all_cv[k]))
print(f'\nOverall best configuration: {best_all}')

In [ ]:
# ── Bar chart comparing all 6 configurations ─────────────────────────────────
cfg_all   = list(all_cv.keys())
means_all = [np.mean(v) for v in all_cv.values()]
stds_all  = [np.std(v)  for v in all_cv.values()]

fig, ax = plt.subplots(figsize=(12, 6))
colours = ['steelblue','seagreen','tomato','goldenrod','orchid','teal']
bars = ax.bar(range(len(cfg_all)), means_all, yerr=stds_all,
              capsize=8, color=colours, width=0.5, alpha=0.85,
              error_kw=dict(elinewidth=2))
ax.set_xticks(range(len(cfg_all)))
ax.set_xticklabels(cfg_all, rotation=15, ha='right', fontsize=9)
ax.set_title('Additional Exercise – All 6 Configurations (5-Fold CV)',
             fontweight='bold')
ax.set_ylabel('Mean Validation Accuracy')
ax.set_ylim(0, 1)
for bar, m, s in zip(bars, means_all, stds_all):
    ax.text(bar.get_x() + bar.get_width()/2, m + s + 0.01,
            f'{m:.3f}\n±{s:.3f}', ha='center', fontsize=7)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('plot_additional_cv.png', dpi=150)
plt.show()
print('Additional CV plot saved.')

### Inference – Additional Exercise

*Comparison:*  
[e.g., C6 (full fine-tune) achieves higher mean accuracy than C5 but at the cost of longer training time and higher SD.  
C5 offers a better accuracy-to-cost trade-off compared to the original C1 baseline while remaining stable.]

*Justification of final choice:*  
[Explain here whether you prefer C4 / C5 / C6 and why, referencing mean accuracy, SD, training time and test accuracy.]

---
## Section 11 – Overall Results Summary

Fill the table below after running all sections.

In [ ]:
# ── Summary table printed to console (copy to results_template.txt) ──────────
print('=' * 65)
print(f'{"Configuration":<25} {"CV Acc":<12} {"SD":<10} {"Test Acc":<12} {"Train Time"}')
print('=' * 65)
rows = [
    ('Baseline (C1)',           '____', '____', '____', '____'),
    ('Best Initialisation',     '____', '____', '____', '____'),
    ('Best Regularisation',     '____', '____', '____', '____'),
    ('Best Optimiser',          '____', '____', '____', '____'),
    ('Best Hyperparameters',    '____', '____', '____', '____'),
    ('Fine-Tuned Model (C4)',   '____', '____', '____', '____'),
]
for row in rows:
    print(f'{row[0]:<25} {row[1]:<12} {row[2]:<10} {row[3]:<12} {row[4]}')
print('=' * 65)
print('\n(Fill values from the outputs above into results_template.txt)')

---
## Section 12 – Discussion Questions

Answer each question below in 3–5 lines. Refer to your experimental results where applicable.

**Q1. What is the difference between model parameters and hyperparameters?**  
> *Parameters* are learned from data (weights, biases). *Hyperparameters* are set before training and control the learning process (LR, batch size, dropout rate).

**Q2. Why is weight initialisation important?**  
> [YOUR ANSWER]

**Q3. Why can zero initialisation be problematic?**  
> All neurons compute identical gradients → the network fails to break symmetry. Every neuron in a layer learns the same feature, making depth useless.

**Q4. Compare Xavier and He initialisation.**  
> Xavier: variance = 2/(fan_in + fan_out) → designed for sigmoid/tanh.  
> He: variance = 2/fan_in → accounts for ReLU's half-zero output; preferred for ReLU networks.

**Q5. How do training/validation curves reveal overfitting?**  
> [YOUR ANSWER]

**Q6. How does Dropout reduce overfitting?**  
> [YOUR ANSWER]

**Q7. What is the purpose of Batch Normalisation?**  
> [YOUR ANSWER]

**Q8. Explain the numerical BN example.**  
> For x = [2,4,6,8]: μ=5, σ²=5, so x̂ ≈ [−1.342, −0.447, 0.447, 1.342]. Output is zero-centred and unit-scaled.

**Q9. What are the roles of γ and β in BN?**  
> γ scales and β shifts the normalised output. They allow BN to represent the identity function (undoing normalisation) if needed, making BN non-destructive.

**Q10. Compare SGD, Momentum, RMSProp and Adam.**  
> [YOUR ANSWER – reference Plot 6 & 7 results]

**Q11. What happens when LR is too large?**  
> [YOUR ANSWER]

**Q12. What happens when LR is too small?**  
> [YOUR ANSWER]

**Q13. Effect of increasing batch size?**  
> [YOUR ANSWER]

**Q14. Explain stride and padding.**  
> *Stride* controls how many pixels the filter moves per step (stride > 1 downsamples the feature map).  
> *Padding* adds zeros around the input to control output size (SAME: output size = input size; VALID: shrinks by K-1).

**Q15. Why is MobileNetV2 computationally efficient?**  
> It uses depthwise separable convolutions (depthwise + 1×1 pointwise) which reduce multiplications from K²·Cin·Cout to K²·Cin + Cin·Cout.

**Q16. What is depthwise separable convolution?**  
> [YOUR ANSWER]

**Q17. What is transfer learning?**  
> [YOUR ANSWER]

**Q18. Differentiate feature extraction and fine-tuning.**  
> [YOUR ANSWER]

**Q19. Why use a smaller LR during fine-tuning?**  
> [YOUR ANSWER]

**Q20. Why is K-Fold CV useful for hyperparameter selection?**  
> [YOUR ANSWER]

**Q21. Why must the test set remain untouched during tuning?**  
> [YOUR ANSWER]

**Q22. Why report both mean and SD?**  
> Mean shows expected performance; SD shows reliability. A model with high mean but high SD is unpredictable on unseen data.

**Q23. Is the highest validation accuracy always sufficient to select a model?**  
> No. Computational cost, training time, variability (SD) and interpretability must also be considered. A marginally lower mean with much lower SD is often more reliable in deployment.

---
## End of Notebook

**Submission checklist:**
- [ ] All 15 plots generated and saved as PNG
- [ ] `results_template.txt` filled with actual values
- [ ] Discussion questions answered (Section 12)
- [ ] Additional exercise completed and justified (Section 10)
- [ ] Notebook runs top-to-bottom without errors

**References:**
1. Goodfellow, Bengio, Courville – *Deep Learning*, MIT Press, 2016.
2. Ioffe & Szegedy – *Batch Normalisation*, ICML 2015.
3. Sandler et al. – *MobileNetV2*, CVPR 2018.
4. Parkhi et al. – *Cats and Dogs*, CVPR 2012.
5. https://www.tensorflow.org | https://keras.io